All'avvio si mette in ascolto dei porti individuati tramite coordinate (che devo ancora controillare).

Quando riceve i dati di una nave che si trova in un porto:

Se è una nave nuova (mai vista prima), la registra. Controlla subito se è tecnicamente ferma (velocità < 0.5 nodi o stato AIS 1 "All'ancora"). Se lo è, deduce che è in coda e fa scattare l'allarme di congestione.

Se è una nave già nota (già nel registro), aggiorna la sua velocità. Poi calcola da quanti minuti è bloccata sottraendo l'ora attuale dal momento in cui l'ha vista la prima volta. Ogni 60 minuti esatti di attesa, stampa un aggiornamento.

In [ ]:
import nest_asyncio
import asyncio
import websockets
import json
import time
from datetime import datetime

nest_asyncio.apply()

API_KEY = "23dff2542eb48c414c4c0213de19b29dd4deaa30"

# Coordinate per le zone di ancoraggio esterne ai principali porti del caffè
PORTS = {
    "Port of Santos (Prevalenza Arabica)": [[-24.15, -46.45], [-23.90, -46.25]],
    "Port of Vitoria (Prevalenza Robusta)": [[-20.45, -40.40], [-20.20, -40.10]],
    "Port of Rio de Janeiro (Arabica/Robusta)": [[-23.05, -43.35], [-22.75, -43.00]],
    "Port of Paranagua (Arabica)": [[-25.65, -48.65], [-25.40, -48.35]],
    "Port of Salvador (Arabica/Robusta)": [[-13.10, -38.70], [-12.70, -38.40]]
}

ais_bounding_boxes = list(PORTS.values())

# Dizionario ottimizzato per tracciare lo stato delle navi in tempo reale
# Struttura: { MMSI: {'port': 'nome', 'first_seen': timestamp, 'status': int, 'sog': float} }
tracked_vessels = {}

# Mappatura degli stati di navigazione AIS rilevanti per i ritardi
NAV_STATUS = {
    0: "In navigazione (Motore acceso)",
    1: "All'ancora (IN ATTESA)",
    5: "Ormeggiata (AL TERMINAL)"
}

def get_port_zone(lat, lon):
    """Funzione ottimizzata per identificare velocemente in quale porto si trova la nave."""
    for port_name, box in PORTS.items():
        if box[0][0] <= lat <= box[1][0] and box[0][1] <= lon <= box[1][1]:
            return port_name
    return None

async def monitor_port_delays():
    async with websockets.connect("wss://stream.aisstream.io/v0/stream") as websocket:

        subscribe_message = {
            "APIKey": API_KEY,
            "BoundingBoxes": ais_bounding_boxes,
            "FilterMessageTypes": ["PositionReport"]
        }

        await websocket.send(json.dumps(subscribe_message))
        print("🟢 Connesso. Analisi congestione porti del caffè avviata...\n" + "-"*70)

        try:
            async for message_json in websocket:
                message = json.loads(message_json)

                if message.get("MessageType") == "PositionReport":
                    data = message["Message"]["PositionReport"]
                    mmsi = data.get("UserID")
                    lat, lon = data.get("Latitude"), data.get("Longitude")
                    sog = data.get("Sog", 0) # Speed Over Ground (Velocità)
                    status_code = data.get("NavigationalStatus", 99)

                    port = get_port_zone(lat, lon)

                    if port:
                        current_time = time.time()

                        # Se è una nuova nave o il suo stato è cambiato, registrala
                        if mmsi not in tracked_vessels:
                            tracked_vessels[mmsi] = {
                                "port": port,
                                "first_seen": current_time,
                                "status": status_code,
                                "sog": sog
                            }

                            status_text = NAV_STATUS.get(status_code, f"Altro ({status_code})")

                            # Alert: Nave bloccata o in attesa all'ancora
                            if status_code == 1 or sog < 0.5:
                                print(f"⚠️ [CONGESTIONE] Nuova nave in coda a {port} (MMSI: {mmsi})")
                                print(f"   Stato: {status_text} | Velocità: {sog} nodi")

                                # Calcolo rapido di quante navi sono attualmente all'ancora in quel porto
                                anchored_count = sum(1 for v in tracked_vessels.values() if v['port'] == port and (v['status'] == 1 or v['sog'] < 0.5))
                                print(f"   📊 Totale navi bloccate/in attesa a {port}: {anchored_count}\n")

                        else:
                            # Se la conoscevamo già, aggiorna i dati e calcola il "Dwell Time" (tempo di attesa)
                            vessel = tracked_vessels[mmsi]
                            vessel["status"] = status_code
                            vessel["sog"] = sog

                            # Se è ancora ferma, calcola da quanti minuti aspetta
                            if status_code == 1 or sog < 0.5:
                                wait_time_minutes = int((current_time - vessel["first_seen"]) / 60)
                                # Stampa un aggiornamento solo ogni ora di attesa (per non intasare lo schermo)
                                if wait_time_minutes > 0 and wait_time_minutes % 60 == 0:
                                    print(f"⏳ [RITARDO] La nave {mmsi} a {port} è in attesa da {wait_time_minutes} minuti.")

        except websockets.exceptions.ConnectionClosed as e:
            print(f"🔴 Connessione chiusa. Codice: {e.code}")
        except Exception as e:
            print(f"⚠️ Errore: {e}")

if __name__ == "__main__":
    asyncio.run(monitor_port_delays())

🟢 Connesso. Analisi congestione porti del caffè avviata...
----------------------------------------------------------------------


KeyboardInterrupt: 

Questo fa esattamente la stessa cosa ma solo 1 volta guardando solo per 10 secondi le aree indicate una voltqa sola poi si spegne.

In [3]:
import nest_asyncio
import asyncio
import websockets
import json
import time

nest_asyncio.apply()

API_KEY = "23dff2542eb48c414c4c0213de19b29dd4deaa30"

# Coordinate per le zone di ancoraggio esterne ai porti
PORTS = {
    "Port of Santos (Prevalenza Arabica)": [[-24.15, -46.45], [-23.90, -46.25]],
    "Port of Vitoria (Prevalenza Robusta)": [[-20.45, -40.40], [-20.20, -40.10]],
    "Port of Rio de Janeiro (Arabica/Robusta)": [[-23.05, -43.35], [-22.75, -43.00]],
    "Port of Paranagua (Arabica)": [[-25.65, -48.65], [-25.40, -48.35]],
    "Port of Salvador (Arabica/Robusta)": [[-13.10, -38.70], [-12.70, -38.40]]
}

ais_bounding_boxes = list(PORTS.values())
tracked_vessels = {}

def get_port_zone(lat, lon):
    for port_name, box in PORTS.items():
        if box[0][0] <= lat <= box[1][0] and box[0][1] <= lon <= box[1][1]:
            return port_name
    return None

async def instant_port_snapshot(listen_time_seconds=10):
    async with websockets.connect("wss://stream.aisstream.io/v0/stream") as websocket:

        subscribe_message = {
            "APIKey": API_KEY,
            "BoundingBoxes": ais_bounding_boxes,
            "FilterMessageTypes": ["PositionReport"]
        }

        await websocket.send(json.dumps(subscribe_message))
        print(f"🟢 Connesso. Raccolta dati in corso per {listen_time_seconds} secondi...\n")

        start_time = time.time()

        try:
            # Ascolta il flusso solo per il tempo prestabilito
            while time.time() - start_time < listen_time_seconds:
                try:
                    # Usiamo un timeout di 1 secondo per non rimanere bloccati se non ci sono messaggi
                    message_json = await asyncio.wait_for(websocket.recv(), timeout=1.0)
                    message = json.loads(message_json)

                    if message.get("MessageType") == "PositionReport":
                        data = message["Message"]["PositionReport"]
                        mmsi = data.get("UserID")
                        lat, lon = data.get("Latitude"), data.get("Longitude")
                        sog = data.get("Sog", 0)
                        status_code = data.get("NavigationalStatus", 99)

                        port = get_port_zone(lat, lon)

                        # Salviamo solo le navi che sono ferme o all'ancora (potenziale congestione)
                        if port and (status_code == 1 or sog < 0.5):
                            if mmsi not in tracked_vessels:
                                tracked_vessels[mmsi] = {
                                    "port": port,
                                    "status": status_code,
                                    "sog": sog
                                }
                except asyncio.TimeoutError:
                    # Nessun messaggio ricevuto in questo secondo, continua il ciclo
                    continue

        except websockets.exceptions.ConnectionClosed as e:
            print(f"🔴 Connessione interrotta prematuramente. Codice: {e.code}")
        except Exception as e:
            print(f"⚠️ Errore: {e}")

        # Genera il report istantaneo alla fine dei 10 secondi
        print("="*60)
        print("REPORT ISTANTANEO CONGESTIONE PORTI BRASILIANI")
        print("="*60)

        for port_name in PORTS.keys():
            # Conta quante navi abbiamo trovato ferme per questo porto
            anchored_ships = sum(1 for v in tracked_vessels.values() if v['port'] == port_name)

            if anchored_ships > 0:
                print(f" {port_name.upper()}: {anchored_ships} navi da carico attualmente bloccate o in attesa.")
            else:
                print(f" {port_name.upper()}: Nessun accumulo rilevato.")

        print("="*60)

if __name__ == "__main__":
    # Esegue lo snapshot (puoi cambiare i secondi se vuoi che raccolga dati per più tempo)
    asyncio.run(instant_port_snapshot(listen_time_seconds=15))

🟢 Connesso. Raccolta dati in corso per 15 secondi...

REPORT ISTANTANEO CONGESTIONE PORTI BRASILIANI
 PORT OF SANTOS (PREVALENZA ARABICA): Nessun accumulo rilevato.
 PORT OF VITORIA (PREVALENZA ROBUSTA): Nessun accumulo rilevato.
 PORT OF RIO DE JANEIRO (ARABICA/ROBUSTA): Nessun accumulo rilevato.
 PORT OF PARANAGUA (ARABICA): Nessun accumulo rilevato.
 PORT OF SALVADOR (ARABICA/ROBUSTA): Nessun accumulo rilevato.


#TEST su porti congestionati

In [2]:
import nest_asyncio
import asyncio
import websockets
import json
import time

nest_asyncio.apply()

# La tua API Key
API_KEY = "23dff2542eb48c414c4c0213de19b29dd4deaa30"

# Coordinate per le zone di ancoraggio dei porti più congestionati al mondo
PORTS = {
    "Porto di Singapore (Asia)": [[1.10, 103.50], [1.35, 104.10]],
    "Porto di Shanghai (Cina)": [[30.50, 121.70], [31.00, 122.30]],
    "Porto di Rotterdam (Europa)": [[51.90, 3.80], [52.10, 4.20]]
}

ais_bounding_boxes = list(PORTS.values())
tracked_vessels = {}

def get_port_zone(lat, lon):
    for port_name, box in PORTS.items():
        if box[0][0] <= lat <= box[1][0] and box[0][1] <= lon <= box[1][1]:
            return port_name
    return None

async def instant_congestion_test(listen_time_seconds=15):
    async with websockets.connect("wss://stream.aisstream.io/v0/stream") as websocket:

        subscribe_message = {
            "APIKey": API_KEY,
            "BoundingBoxes": ais_bounding_boxes,
            "FilterMessageTypes": ["PositionReport"]
        }

        await websocket.send(json.dumps(subscribe_message))
        print(f"🟢 Connesso. In ascolto sui mega-hub globali per {listen_time_seconds} secondi...\n")

        start_time = time.time()

        try:
            while time.time() - start_time < listen_time_seconds:
                try:
                    message_json = await asyncio.wait_for(websocket.recv(), timeout=1.0)
                    message = json.loads(message_json)

                    if message.get("MessageType") == "PositionReport":
                        data = message["Message"]["PositionReport"]
                        mmsi = data.get("UserID")
                        lat, lon = data.get("Latitude"), data.get("Longitude")
                        sog = data.get("Sog", 0)
                        status_code = data.get("NavigationalStatus", 99)

                        port = get_port_zone(lat, lon)

                        # Filtriamo solo le navi ferme o all'ancora
                        if port and (status_code == 1 or sog < 0.5):
                            if mmsi not in tracked_vessels:
                                tracked_vessels[mmsi] = {
                                    "port": port,
                                    "status": status_code,
                                    "sog": sog
                                }
                except asyncio.TimeoutError:
                    continue

        except websockets.exceptions.ConnectionClosed as e:
            print(f"🔴 Connessione interrotta prematuramente. Codice: {e.code}")
        except Exception as e:
            print(f"⚠️ Errore: {e}")

        # Genera il report finale
        print("="*60)
        print("REPORT ISTANTANEO: MEGA-PORTI GLOBALI")
        print("="*60)

        for port_name in PORTS.keys():
            anchored_ships = sum(1 for v in tracked_vessels.values() if v['port'] == port_name)

            if anchored_ships > 0:
                print(f" {port_name.upper()}: {anchored_ships} navi in attesa o bloccate rilevate.")
            else:
                print(f" {port_name.upper()}: Nessun accumulo rilevato.")

        print("="*60)

if __name__ == "__main__":
    # Ho aumentato il tempo a 15 secondi per darti un campione ancora più ampio
    asyncio.run(instant_congestion_test(listen_time_seconds=15))

🟢 Connesso. In ascolto sui mega-hub globali per 15 secondi...

REPORT ISTANTANEO: MEGA-PORTI GLOBALI
 PORTO DI SINGAPORE (ASIA): Nessun accumulo rilevato.
 PORTO DI SHANGHAI (CINA): Nessun accumulo rilevato.
 PORTO DI ROTTERDAM (EUROPA): Nessun accumulo rilevato.
